# Muon vs SGD: Staged Linear Network

Consider a two-layer linear network $\hat y = W_2 W_1 x$, with the target $W_2 W_1 \approx A$.

**Target decomposition.** Take the SVD of $A$: $A = U \Sigma V^\top$, and define
$$W_1^* = \sqrt{\Sigma}\, V^\top, \quad W_2^* = U \sqrt{\Sigma}$$
so that $W_2^* W_1^* = A$. In Phase 1, only $W_1$ is trained, using $\|W_1 - W_1^*\|$ as the loss. In Phase 2, only $W_2$ is trained, using the composition error $\|W_2 W_1 - A\|$ as the loss.

**Staged training**
- Phase 1 (SGD vs Muon): freeze $W_2$ and train $W_1$ with either SGD or Muon, minimizing $\|W_1 - W_1^*\|^2 / \|A\|^2$.
- Phase 2 (SGD only): freeze the $W_1$ obtained from Phase 1, reinitialize $W_2$, and use only SGD to minimize $\|W_2 W_1 - A\|^2 / \|A\|^2$ until $\|W_2 W_1 - A\| / \|A\| \le \text{THRESHOLD}$.

Muon uses the same learning rate as SGD. At each step, it applies NS orthogonalization to the current gradient, with update norm lr $\|g\|$.

In [ ]:
import math

import torch

torch.set_default_dtype(torch.float64)

D = 64
PHASE1_LR = 2e-1
PHASE2_LR = 2e-2
SEED = 0
THRESHOLD = 0.05
PHASE1_STEPS = 600
PHASE2_MAX_STEPS = 20000

In [ ]:
def zeropower_via_newtonschulz5(G, steps=3, eps=1e-7):
    assert len(G.shape) == 2
    a, b, c = (3.4445, -4.7750, 2.0315)
    X = G.bfloat16()
    X = X / (X.norm() + eps)
    if G.size(0) > G.size(1):
        X = X.T
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * A @ A
        X = a * X + B @ X
    if G.size(0) > G.size(1):
        X = X.T
    return X.to(dtype=G.dtype)


def muon_direction(g, eps=1e-7):
    direction = zeropower_via_newtonschulz5(g)
    return direction * (g.norm() / (direction.norm() + eps))


class Muon(torch.optim.Optimizer):

    def __init__(self, params, lr=1e-3):
        super().__init__(params, dict(lr=lr))

    def step(self):
        for group in self.param_groups:
            lr = group["lr"]
            for p in group["params"]:
                g = p.grad
                if g is None:
                    continue
                p.data.add_(muon_direction(g), alpha=-lr)

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_target(d, seed=0):
    gen = torch.Generator().manual_seed(seed)
    return torch.randn(d, d, generator=gen) / math.sqrt(d)


def svd_factors(A):
    u, s, vh = torch.linalg.svd(A.cpu(), full_matrices=False)
    sqrt_s = torch.sqrt(s)
    w1_star = (torch.diag(sqrt_s) @ vh).cuda()
    w2_star = (u @ torch.diag(sqrt_s)).cuda()
    return w1_star, w2_star


def rel_dist(M, target, scale):
    return (M - target).norm().item() / (scale + 1e-12)


def compose_loss(W2, W1, A, norm_A):
    M = W2 @ W1 - A
    return (M ** 2).sum() / (norm_A**2)


def steps_to(values, threshold):
    for i, v in enumerate(values):
        if v <= threshold:
            return i
    return None


def rand_matrix(d, seed):
    state = torch.cuda.get_rng_state()
    torch.cuda.manual_seed(seed)
    W = torch.randn(d, d, device="cuda") / math.sqrt(d)
    torch.cuda.set_rng_state(state)
    return W


def w1_spectrum_flatness(W1):
    s = torch.linalg.svdvals(W1)
    return (s.pow(2).sum() / s[0].pow(2)).item() / D

In [ ]:
def train_staged(
    A,
    w1_star,
    w2_star,
    phase1_optimizer_class,
    phase1_steps=PHASE1_STEPS,
    phase2_max_steps=PHASE2_MAX_STEPS,
    phase2_threshold=THRESHOLD,
    phase2_lr=PHASE2_LR,
    seed=SEED,
):
    set_seed(seed)
    W1 = rand_matrix(D, seed)
    norm_A = A.norm().item()
    w1_star = w1_star.detach()

    w1_dist = []

    W1.requires_grad_(True)
    opt1 = phase1_optimizer_class([W1], lr=PHASE1_LR)
    for step in range(phase1_steps + 1):
        w1_dist.append(rel_dist(W1.detach(), w1_star, norm_A))
        if step == phase1_steps:
            break
        loss = ((W1 - w1_star) ** 2).sum() / (norm_A**2)
        opt1.zero_grad(set_to_none=True)
        loss.backward()
        opt1.step()

    W1_frozen = W1.detach().clone()

    W2 = rand_matrix(D, seed + 1)
    W2.requires_grad_(True)
    opt2 = torch.optim.SGD([W2], lr=phase2_lr)
    tot_dist, w2_dist = [], []
    phase2_steps = 0
    for step in range(phase2_max_steps + 1):
        W2d = W2.detach()
        dist = rel_dist(W2d @ W1_frozen, A, norm_A)
        tot_dist.append(dist)
        w2_dist.append(rel_dist(W2d, w2_star, norm_A))
        phase2_steps = step
        if dist <= phase2_threshold:
            break
        if step == phase2_max_steps:
            break
        loss = compose_loss(W2, W1_frozen, A, norm_A)
        opt2.zero_grad(set_to_none=True)
        loss.backward()
        opt2.step()

    return dict(
        tot_dist=tot_dist,
        w1_dist=w1_dist,
        w2_dist=w2_dist,
        W1=W1_frozen,
        W2=W2.detach(),
        phase1_steps=phase1_steps,
        phase2_steps=phase2_steps,
    )


def run_staged_experiment(A, w1_star, w2_star):
    print("\n" + "=" * 72)
    print(f"Staged training: phase1 ({PHASE1_STEPS} steps), phase2 (threshold {THRESHOLD})")
    print(f"  phase1 lr={PHASE1_LR}, loss = ||W1-W1*||^2/||A||^2")
    print(f"  phase2 lr={PHASE2_LR}, loss = ||W2 W1 - A||^2/||A||^2")
    print("=" * 72)
    sgd = train_staged(A, w1_star, w2_star, torch.optim.SGD)
    muon = train_staged(A, w1_star, w2_star, Muon)
    norm_A = A.norm().item()

    for label, run in [("SGD W1 -> SGD W2", sgd), ("Muon W1 -> SGD W2", muon)]:
        p1 = run["phase1_steps"]
        print(f"\n  [{label}]")
        print(f"    phase1 step {p1}: ||W1-W1*||={run['w1_dist'][-1]:.4f}")
        print(f"    phase2 step {p1 + run['phase2_steps']}: ||W1-W1*||={rel_dist(run['W1'], w1_star, norm_A):.4f}, "
              f"||W2-W2*||={run['w2_dist'][-1]:.4f}, ||W2 W1 - A||/||A||={run['tot_dist'][-1]:.6f}, "
              f"steps={run['phase2_steps']}")
        p2_hit = steps_to(run["tot_dist"], THRESHOLD)
        total = p1 + p2_hit if p2_hit is not None else None
        print(f"    steps@{THRESHOLD} total={total}")

    sgd_p2 = steps_to(sgd["tot_dist"], THRESHOLD)
    muon_p2 = steps_to(muon["tot_dist"], THRESHOLD)
    delta = sgd_p2 - muon_p2 if sgd_p2 and muon_p2 else None
    print(f"\n  delta={delta} (positive => Muon faster)")

    print(f"\n  Phase1 end W1 spectrum flatness (higher => flatter):")
    for label, run in [("SGD W1", sgd), ("Muon W1", muon)]:
        flat = w1_spectrum_flatness(run["W1"])
        print(f"    {label}: {flat:.4f}")
    flat_s = w1_spectrum_flatness(sgd["W1"])
    flat_m = w1_spectrum_flatness(muon["W1"])
    print(f"    Muon - SGD: {flat_m - flat_s:+.4f}")

    return sgd, muon

In [ ]:
set_seed(SEED)
A = make_target(D, seed=SEED).cuda()
w1_star, w2_star = svd_factors(A)
sgd_run, muon_run = run_staged_experiment(A, w1_star, w2_star)

## Summary

Muon ends Phase 1 farther from $W_1^*$, but the spectrum of its $W_1$ is more favorable for Phase 2. Net effect: Phase 2 may require fewer steps, and the overall optimization process can still be faster (`delta > 0`).

Why can Muon be faster in Phase 2?

Phase 2 is equivalent to fixing the encoder $W_1$ and learning a linear probe $W_2$ such that $W_2 W_1 \approx A$. Let $E_t = W_{2,t}W_1 - A$. Then
$$L(W_2)=\tfrac12\|W_2 W_1 - A\|_F^2,\qquad \nabla_{W_2}L=(W_2 W_1 - A)W_1^\top$$
The SGD update is $W_{2,t+1}=W_{2,t}-\eta(W_{2,t}W_1-A)W_1^\top$, which gives the error recurrence
$$E_{t+1}=E_t(I-\eta W_1^\top W_1)$$
In the eigenbasis of $W_1^\top W_1$, the $i$-th singular value $s_i(E_t)$ of $E_t$ satisfies $s_i(E_{t+1})=|1-\eta\sigma_i^2|\,s_i(E_t)$, where $\sigma_i$ is the $i$-th singular value of $W_1$. When $\eta=\alpha/\sigma_{\max}(W_1)^2$, the slowest contraction factor is $1-\alpha/\kappa(W_1)^2$.

Therefore, the closer $W_1$ is to orthogonal (flatter spectrum, smaller condition number), the faster Phase 2 optimizes.

Above, the loss is constructed directly from $W_2 W_1 - A$. In practice, a more common setup is to sample $x$, compute the output $y=W_2 W_1 x$, and compare it with the target $Ax$.

One can show that the closer $W_1$ is to orthogonal, the easier it is for a linear probe to recover the original input $x$ from the intermediate representation $h=W_1 x$. This is the same reason that learning $Ax$ becomes easier. To recover $x$, learn a probe $B$ such that $B W_1 \approx I$, with loss
$$L_x(B)=\tfrac12\|B W_1 - I\|_F^2$$
To recover $Ax$, simply replace the target matrix with $A$:
$$L_A(B)=\tfrac12\|B W_1 - A\|_F^2$$
Let $E_t = B_t W_1 - M$ ($M=I$ or $M=A$). We again have $E_{t+1} = E_t(I - \eta W_1^\top W_1)$, so the convergence speed is still determined only by the singular values of $W_1$.

## Interpreting Muon

In summary, Muon's advantage can be understood as follows. Each upstream update plays two roles at once: first, it reduces the current loss along the negative-gradient direction so training can continue; second, it acts like a temporary residual connection, preserving as much input information as possible in the output and passing it downstream. Because it is not known in advance whether the downstream module will need $x$ or $W_1 x$ more, encoding $x$ through an orthogonal transformation is the most favorable choice for downstream linear recovery of $x$. Muon's trade-off is exactly to orthogonalize the gradient before using it as the update direction. For the Phase 1 subproblem, this is no longer the steepest descent direction under that loss. But the resulting $W_1$ provides a better representation for the downstream module, making Phase 2 easier. Once Phase 2 optimizes better, it can in turn provide Phase 1 with an easier subproblem, compensating for Muon's weaker single-subproblem solving ability in Phase 1 and ultimately making the overall optimization faster.

## Extension: black-box downstream with an explicit Jacobian

For a more general downstream module, suppose Phase 1 still freezes $W_1$, but Phase 2 trains a differentiable black-box map
$$
F_\theta(W_1) \approx A.
$$
Here $F_\theta$ can be nonlinear and does not need to expose a simple matrix factorization. The only object we need is the Jacobian of the flattened prediction with respect to the downstream parameters:
$$
e_t=\operatorname{vec}(F_{\theta_t}(W_1)-A),
\qquad
J_t=\frac{\partial e_t}{\partial \theta_t}.
$$
For the normalized squared loss
$$
L(\theta)=\frac{1}{2\|A\|_F^2}\|e(\theta)\|_2^2,
$$
GD gives
$$
\theta_{t+1}=\theta_t-\frac{\eta}{\|A\|_F^2}J_t^\top e_t.
$$
The one-step prediction comes from a first-order Taylor expansion. Around $\theta_t$,
$$
e(\theta_{t+1})
\approx
e(\theta_t)+J_t(\theta_{t+1}-\theta_t).
$$
Substituting the GD update gives
$$
e_{t+1}
\approx
e_t-\frac{\eta}{\|A\|_F^2}J_tJ_t^\top e_t
=
\left(I-\frac{\eta}{\|A\|_F^2}J_tJ_t^\top\right)e_t.
$$
Thus the black-box analogue of the previous exponential formula is controlled by the Jacobian Gram matrix
$$
K_t=J_tJ_t^\top.
$$
The next approximation is about the norm of this update. Let
$$
\rho_t=\frac{\eta}{\|A\|_F^2}\frac{e_t^\top K_t e_t}{e_t^\top e_t}.
$$
Expanding the squared norm of the one-step prediction gives
$$
\|e_{t+1}\|_2^2
\approx
\|e_t\|_2^2
-2\frac{\eta}{\|A\|_F^2}e_t^\top K_t e_t
+O(\eta^2).
$$
Ignoring the $O(\eta^2)$ term, we get
$$
\frac{\|e_{t+1}\|_2}{\|e_t\|_2}
\approx
\sqrt{1-2\rho_t}
\approx
1-\rho_t,
$$
where the last step uses $\sqrt{1-z}\approx 1-z/2$ for small $z$. Taking a log and using $\log(1-\rho_t)\approx-\rho_t$ gives the local estimate
$$
\log\frac{\|e_{t+1}\|_2}{\|e_t\|_2}
\approx
-\frac{\eta}{\|A\|_F^2}\frac{e_t^\top K_t e_t}{e_t^\top e_t}.
$$
This is a local formula: it is accurate when the GD step is small enough that the Taylor expansion is reliable and the $O(\eta^2)$ terms are negligible.

For many steps, summing the log-ratio estimates gives
$$
\log\frac{\|e_t\|_2}{\|e_0\|_2}
\approx
-\frac{\eta}{\|A\|_F^2}\sum_{s<t}\lambda_{\mathrm{eff},s},
\qquad
\lambda_{\mathrm{eff},s}=\frac{e_s^\top K_s e_s}{e_s^\top e_s}.
$$
Exponentiating both sides gives
$$
\|e_t\|_2
\approx
\exp\left(-\frac{\eta}{\|A\|_F^2}\sum_{s<t}\lambda_{\mathrm{eff},s}\right)\|e_0\|_2.
$$
If $K_t$ changes slowly, this multi-step approximation is easier to interpret as a smooth exponential decay with a time-varying effective rate.
When the downstream is linear in its parameters, this reduces to the exact linear recurrence used earlier. For a nonlinear black-box, it is a local prediction; the mismatch measures Jacobian drift and higher-order terms.


This also explains how the black-box formula relates to the earlier linear-probe formula. The absence of an explicit $W_1^\top W_1$ term does not mean that $W_1$ no longer matters. Rather, $W_1$ is now inside the Jacobian:
$$
J_t = J_t(W_1)=\frac{\partial\,\operatorname{vec}(F_{\theta_t}(W_1)-A)}{\partial \theta_t}.
$$
Changing the upstream representation changes the sensitivity of the black-box output to its parameters, and therefore changes $K_t=J_t(W_1)J_t(W_1)^\top$.

The one-layer downstream case is the simplest special case of this Jacobian view. If
$$
F_\theta(W_1)=BW_1,
$$
where the downstream parameter is $\theta=\operatorname{vec}(B)$, then the Jacobian Gram operator acts as
$$
K(E)=E W_1^\top W_1.
$$
Thus the general Jacobian recurrence reduces exactly to
$$
E_{t+1}=E_t\left(I-\frac{\eta}{\|A\|_F^2}W_1^\top W_1\right)
$$
for the normalized loss used in code. In the linear probe, the role of $W_1$ is visible as $W_1^\top W_1$; in a nonlinear black-box, the same role is implicit in the conditioning and effective eigenvalues of $J_t(W_1)J_t(W_1)^\top$.


In [ ]:
BB_HIDDEN = 64
BB_LR = 8e-2
BB_MAX_STEPS = 15000
BB_THRESHOLD = 0.05
BB_CHECK_EVERY = 5000


def init_blackbox_params(seed=SEED + 10):
    set_seed(seed)
    W_hidden = (torch.randn(BB_HIDDEN, D, device="cuda") / math.sqrt(D)).requires_grad_(True)
    W_out = (torch.randn(D, BB_HIDDEN, device="cuda") / math.sqrt(BB_HIDDEN)).requires_grad_(True)
    return [W_hidden, W_out]


def blackbox_forward(params, W1):
    W_hidden, W_out = params
    return W_out @ torch.tanh(W_hidden @ W1)


def flatten_params(params):
    return torch.cat([p.reshape(-1) for p in params])


def unflatten_blackbox(theta):
    hidden_size = BB_HIDDEN * D
    W_hidden = theta[:hidden_size].reshape(BB_HIDDEN, D)
    W_out = theta[hidden_size:].reshape(D, BB_HIDDEN)
    return [W_hidden, W_out]


def jacobian_and_error_from_theta(theta_anchor, W1, A):
    theta = theta_anchor.detach().requires_grad_(True)

    def error_from_theta(theta_value):
        return (blackbox_forward(unflatten_blackbox(theta_value), W1) - A).reshape(-1)

    error = error_from_theta(theta).detach()
    jacobian = torch.autograd.functional.jacobian(error_from_theta, theta, vectorize=True).detach()
    return jacobian, error


def jacobian_and_error(params, W1, A):
    return jacobian_and_error_from_theta(flatten_params([p.detach() for p in params]), W1, A)


def train_blackbox_downstream(
    W1,
    A,
    lr=BB_LR,
    max_steps=BB_MAX_STEPS,
    threshold=BB_THRESHOLD,
    check_every=BB_CHECK_EVERY,
    record_every=1,
    with_jacobian_checks=False,
):
    params = init_blackbox_params()
    opt = torch.optim.SGD(params, lr=lr)
    norm_A = A.norm().item()
    curve, logs = [], []
    hits = {BB_THRESHOLD: None}

    for step in range(max_steps + 1):
        prediction = blackbox_forward(params, W1)
        error_matrix = prediction - A
        rel_error = error_matrix.norm().item() / norm_A
        if step % record_every == 0:
            curve.append((step, rel_error))
        for th in hits:
            if hits[th] is None and rel_error <= th:
                hits[th] = step

        if with_jacobian_checks and (step % check_every == 0 or rel_error <= threshold or step == max_steps):
            J, e = jacobian_and_error(params, W1, A)
            K = J @ J.T
            scaled_lr = lr / (norm_A**2)
            pred_next_e = e - scaled_lr * (K @ e)
            pred_next_rel = pred_next_e.norm().item() / norm_A
            lambda_eff = (e @ (K @ e) / (e @ e)).item()
            logs.append(
                dict(
                    step=step,
                    rel_error=rel_error,
                    pred_next_rel=pred_next_rel,
                    actual_next_rel=None,
                    lambda_eff=lambda_eff,
                )
            )

        if rel_error <= threshold or step == max_steps:
            break

        loss = 0.5 * (error_matrix ** 2).sum() / (norm_A**2)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

        if logs and logs[-1]["step"] == step:
            with torch.no_grad():
                logs[-1]["actual_next_rel"] = (blackbox_forward(params, W1) - A).norm().item() / norm_A

    return dict(curve=curve, logs=logs, hits=hits, final_step=step, final_error=rel_error)


def run_blackbox_jacobian_experiment():
    print("\n" + "=" * 72)
    print("Black-box downstream: D=64 main Phase-1 W1, predicting GD with J J^T")
    print(f"  hidden={BB_HIDDEN}, lr={BB_LR}, threshold={BB_THRESHOLD}")
    print("=" * 72)

    results = {}
    for label, run in [("SGD-trained W1", sgd_run), ("Muon-trained W1", muon_run)]:
        result = train_blackbox_downstream(run["W1"], A, with_jacobian_checks=True)
        results[label] = result
        one_step_errors = [
            abs(row["actual_next_rel"] - row["pred_next_rel"])
            for row in result["logs"]
            if row["actual_next_rel"] is not None
        ]
        print(f"\n  [{label}]")
        print(f"    phase1 ||W1-W1*||={run['w1_dist'][-1]:.4f}, flatness={w1_spectrum_flatness(run['W1']):.4f}")
        print(f"    hits: {result['hits']}, final step={result['final_step']}, final error={result['final_error']:.6f}")
        for row in result["logs"]:
            print(
                f"    step {row['step']:5d}: rel={row['rel_error']:.6f}, "
                f"pred_next={row['pred_next_rel']:.6f}, actual_next={row['actual_next_rel']}, "
                f"lambda_eff={row['lambda_eff']:.3e}"
            )
        print(
            f"    one-step prediction abs error: mean={sum(one_step_errors) / len(one_step_errors):.3e}, "
            f"max={max(one_step_errors):.3e}"
        )

    s = results["SGD-trained W1"]["hits"][BB_THRESHOLD]
    m = results["Muon-trained W1"]["hits"][BB_THRESHOLD]
    delta = s - m if s is not None and m is not None else None
    print(f"\n  delta@{BB_THRESHOLD}={delta} (positive => Muon-trained W1 faster)")
    return results


blackbox_jacobian_results = run_blackbox_jacobian_experiment()


### Multi-step Jacobian prediction

The previous check was a one-step prediction. To test the formula over many GD steps, there are two natural approximations:

1. **Frozen-kernel prediction:** compute $K_0=J_0J_0^\top$ once and repeatedly apply
$$
\hat e_{t+1}=\left(I-\frac{\eta}{\|A\|_F^2}K_0\right)\hat e_t.
$$
This is exact only if the downstream map is linear in parameters, or if the Jacobian stays nearly constant.

2. **Rolling-kernel prediction:** every few steps, recompute $K_t=J_tJ_t^\top$ on the actual trajectory, then use that local kernel to predict the next short segment. This tests whether the local Jacobian formula remains predictive when the black-box Jacobian drifts during training.

The plot below compares both predictions with the actual GD curve. The fixed $K_0$ curve captures the overall trend but drifts over long horizons; the rolling Jacobian prediction stays close because it updates the local linearization.


In [ ]:
import matplotlib.pyplot as plt

ROLLING_INTERVAL = 2500
BLACKBOX_MULTISTEP_FIG = "blackbox_jacobian_multistep_prediction.png"


def train_blackbox_with_snapshots(W1, A, max_steps=BB_MAX_STEPS, threshold=BB_THRESHOLD, rolling_interval=ROLLING_INTERVAL):
    params = init_blackbox_params()
    opt = torch.optim.SGD(params, lr=BB_LR)
    norm_A = A.norm().item()
    actual_errors, snapshots = [], []

    for step in range(max_steps + 1):
        with torch.no_grad():
            rel_error = (blackbox_forward(params, W1) - A).norm().item() / norm_A
            actual_errors.append(rel_error)
        if step % rolling_interval == 0 or rel_error <= threshold or step == max_steps:
            snapshots.append((step, flatten_params(params)))
        if rel_error <= threshold or step == max_steps:
            break

        error_matrix = blackbox_forward(params, W1) - A
        loss = 0.5 * (error_matrix ** 2).sum() / (norm_A**2)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

    return torch.tensor(actual_errors), snapshots


def frozen_kernel_prediction(theta_anchor, W1, A, horizon):
    J, e = jacobian_and_error_from_theta(theta_anchor, W1, A)
    K = J @ J.T
    scaled_lr = BB_LR / (A.norm().item() ** 2)
    pred_errors = []
    current_e = e.clone()
    norm_A = A.norm().item()

    for _ in range(horizon + 1):
        pred_errors.append(current_e.norm().item() / norm_A)
        current_e = current_e - scaled_lr * (K @ current_e)

    lambda_eff = (e @ (K @ e) / (e @ e)).item()
    return torch.tensor(pred_errors), lambda_eff


def rolling_kernel_prediction(W1, A, snapshots, total_steps):
    pred_errors = torch.empty(total_steps + 1)
    lambda_eff_by_anchor = []
    for idx, (start, theta_anchor) in enumerate(snapshots[:-1]):
        end = snapshots[idx + 1][0]
        segment_pred, lambda_eff = frozen_kernel_prediction(theta_anchor, W1, A, end - start)
        pred_errors[start : end + 1] = segment_pred
        lambda_eff_by_anchor.append((start, lambda_eff))
    return pred_errors, lambda_eff_by_anchor


def blackbox_multistep_case(label, W1):
    actual, snapshots = train_blackbox_with_snapshots(W1, A)
    total_steps = len(actual) - 1
    frozen_from_start, lambda_eff_0 = frozen_kernel_prediction(snapshots[0][1], W1, A, total_steps)
    rolling, lambda_eff_by_anchor = rolling_kernel_prediction(W1, A, snapshots, total_steps)
    return dict(
        label=label,
        actual=actual.cpu(),
        frozen=frozen_from_start.cpu(),
        rolling=rolling.cpu(),
        lambda_eff_0=lambda_eff_0,
        lambda_eff_by_anchor=lambda_eff_by_anchor,
    )


def run_blackbox_multistep_prediction_plot():
    cases = [
        blackbox_multistep_case("SGD-trained W1", sgd_run["W1"]),
        blackbox_multistep_case("Muon-trained W1", muon_run["W1"]),
    ]

    print("\n" + "=" * 72)
    print("Black-box downstream: multi-step Jacobian prediction")
    print(f"  fixed K0 vs rolling K_t every {ROLLING_INTERVAL} steps")
    print("=" * 72)
    for case in cases:
        frozen_mae = (case["actual"] - case["frozen"]).abs().mean().item()
        rolling_mae = (case["actual"] - case["rolling"]).abs().mean().item()
        print(f"\n  [{case['label']}]")
        print(
            f"    final actual/frozen/rolling = "
            f"{case['actual'][-1]:.6f} / {case['frozen'][-1]:.6f} / {case['rolling'][-1]:.6f}"
        )
        print(f"    mean abs curve error: frozen={frozen_mae:.3e}, rolling={rolling_mae:.3e}")
        print(f"    lambda_eff at step 0: {case['lambda_eff_0']:.3e}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
    for ax, case in zip(axes, cases):
        steps = range(len(case["actual"]))
        ax.plot(steps, case["actual"], label="actual GD", lw=2.2, color="#243447")
        ax.plot(steps, case["frozen"], label="multi-step prediction, fixed K0", lw=1.8, ls="--", color="#6C63FF")
        ax.plot(
            steps,
            case["rolling"],
            label=f"rolling prediction, K every {ROLLING_INTERVAL} steps",
            lw=1.8,
            ls=":",
            color="#E83E8C",
        )
        ax.set_title(case["label"])
        ax.set_xlabel("downstream GD step")
        ax.grid(True, alpha=0.25)
    axes[0].set_ylabel(r"relative error $\|F_\theta(W_1)-A\|/\|A\|$")
    axes[1].legend(frameon=False, loc="upper right")
    fig.tight_layout()
    fig.savefig(BLACKBOX_MULTISTEP_FIG, dpi=180)
    plt.show()
    return cases


blackbox_multistep_results = run_blackbox_multistep_prediction_plot()


### Full black-box downstream training curves

The Jacobian-prediction plots above include the actual GD curves, but the comparison is easier to read if we plot only the actual black-box downstream training curves on the same axes. Here $W_1$ is not hand-crafted: it is exactly the $D=64$ upstream matrix obtained from the main Phase 1 run, using either SGD or Muon with the same Phase 1 settings as above.


In [ ]:
BLACKBOX_ACTUAL_CURVES_FIG = "blackbox_actual_training_curves.png"


def run_blackbox_actual_curve_plot():
    sgd_result = train_blackbox_downstream(sgd_run["W1"], A, with_jacobian_checks=False)
    muon_result = train_blackbox_downstream(muon_run["W1"], A, with_jacobian_checks=False)

    print("\n" + "=" * 72)
    print("Black-box downstream: actual training curves")
    print("=" * 72)
    print(
        f"  SGD-trained W1:  phase1 dist={sgd_run['w1_dist'][-1]:.4f}, "
        f"flatness={w1_spectrum_flatness(sgd_run['W1']):.4f}, hits={sgd_result['hits']}"
    )
    print(
        f"  Muon-trained W1: phase1 dist={muon_run['w1_dist'][-1]:.4f}, "
        f"flatness={w1_spectrum_flatness(muon_run['W1']):.4f}, hits={muon_result['hits']}"
    )

    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    ax.plot([x for x, _ in sgd_result["curve"]], [y for _, y in sgd_result["curve"]], label="SGD-trained W1", lw=2.2, color="#2F80ED")
    ax.plot([x for x, _ in muon_result["curve"]], [y for _, y in muon_result["curve"]], label="Muon-trained W1", lw=2.2, color="#D946EF")
    ax.axhline(BB_THRESHOLD, color="#243447", lw=1.0, ls=":", alpha=0.65)
    ax.set_xlabel("downstream GD step")
    ax.set_ylabel(r"relative error $\|F_\theta(W_1)-A\|/\|A\|$")
    ax.set_title("Black-box downstream actual training curves")
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(BLACKBOX_ACTUAL_CURVES_FIG, dpi=180)
    plt.show()
    return {"sgd": sgd_result, "muon": muon_result}


blackbox_actual_curve_results = run_blackbox_actual_curve_plot()
